# CareBridge hospital occupancy forecasting
## Reproducible training, validation and model-card notebook
This notebook documents the complete hackathon ML workflow. The default dataset is deterministic, privacy-safe simulation. It demonstrates engineering quality but is **not clinically validated** and must not drive patient placement without governed hospital data and human confirmation.

## 1. Objectives
- Forecast hospital occupancy over the next operational interval.
- Compare the model with a persistence baseline.
- Use a chronological split to reduce future-data leakage.
- inspect errors by facility type and occupancy band.
- Export versioned `.pkl` and `.joblib` artifacts for the API.

In [ ]:
from pathlib import Path
import json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from train import (generate_simulated_data, prepare_features, train_model, FEATURE_COLUMNS, MODEL_PATH, METRICS_PATH)
pd.set_option('display.max_columns', 30)
ROOT = Path.cwd()


## 2. Data provenance and governance
The model's target requires hourly facility-level snapshots. Public hospital directories provide identity and location, not live occupancy. Real deployment data must come from participating hospitals under a data-sharing agreement. Never include patient names, ABHA identifiers, phone numbers, diagnoses or free-text notes.

In [ ]:
sources = json.loads((ROOT / 'data_sources.json').read_text())
pd.DataFrame(sources['sources'])[['name','publisher','role']]


## 3. Generate the deterministic demonstration dataset
Forty facilities are simulated over 120 days. The random seed is fixed, making training and tests reproducible.

In [ ]:
data = generate_simulated_data(hospitals=40, days=120, seed=42)
print(f'Rows: {len(data):,} | Hospitals: {data.hospital_id.nunique()}')
data.head()


## 4. Schema and quality gates

In [ ]:
required = FEATURE_COLUMNS + ['timestamp','hospital_id','occupancy_next_hour']
assert set(required).issubset(data.columns)
assert data[FEATURE_COLUMNS + ['occupancy_next_hour']].isna().sum().sum() == 0
assert data['current_occupancy'].between(0,100).all()
assert data['occupancy_next_hour'].between(0,100).all()
data[required].dtypes


In [ ]:
quality = pd.DataFrame({'missing': data[required].isna().sum(), 'unique': data[required].nunique()})
quality


## 5. Exploratory analysis

In [ ]:
data[['current_occupancy','occupancy_next_hour','temperature']].describe(percentiles=[.01,.1,.25,.5,.75,.9,.99])


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,4))
data.current_occupancy.hist(bins=30, ax=axes[0], color='#0b7a70')
axes[0].set(title='Current occupancy distribution', xlabel='Occupancy %', ylabel='Rows')
data.groupby('hour').occupancy_next_hour.mean().plot(ax=axes[1], marker='o', color='#e85d3f')
axes[1].set(title='Mean target by hour', ylabel='Next-hour occupancy %')
plt.tight_layout()


In [ ]:
facility_summary = data.groupby('facility_type').agg(rows=('hospital_id','size'), hospitals=('hospital_id','nunique'), mean_occupancy=('current_occupancy','mean'), p90_occupancy=('current_occupancy',lambda s:s.quantile(.9)))
facility_summary.round(2)


## 6. Leakage-safe chronological split
A random split would allow later observations from a hospital to influence training for earlier test observations. We instead reserve the latest 20% of timestamps.

In [ ]:
cutoff = data.timestamp.quantile(.80)
train = data[data.timestamp < cutoff].copy()
test = data[data.timestamp >= cutoff].copy()
assert train.timestamp.max() < test.timestamp.min()
{'cutoff': cutoff, 'train_rows': len(train), 'test_rows': len(test)}


## 7. Train the production pipeline
The pipeline combines one-hot encoding for facility type with a regularized histogram gradient-boosting regressor. Training writes both supported artifact formats and a metrics manifest.

In [ ]:
metrics = train_model()
metrics


In [ ]:
artifact = joblib.load(MODEL_PATH)
model = artifact['model']
predicted = np.clip(model.predict(test[FEATURE_COLUMNS]),0,100)
baseline = test.current_occupancy.to_numpy()
evaluation = pd.Series({'model_mae':mean_absolute_error(test.occupancy_next_hour,predicted), 'baseline_mae':mean_absolute_error(test.occupancy_next_hour,baseline), 'rmse':mean_squared_error(test.occupancy_next_hour,predicted)**.5, 'r2':r2_score(test.occupancy_next_hour,predicted)})
evaluation.round(3)


## 8. Error diagnostics

In [ ]:
diagnostics = test[['hospital_id','facility_type','current_occupancy','occupancy_next_hour']].copy()
diagnostics['prediction'] = predicted
diagnostics['absolute_error'] = (diagnostics.occupancy_next_hour-diagnostics.prediction).abs()
diagnostics['occupancy_band'] = pd.cut(diagnostics.current_occupancy,[0,50,70,85,100],include_lowest=True)
diagnostics.groupby('occupancy_band',observed=True).absolute_error.agg(['count','mean','median','max']).round(3)


In [ ]:
diagnostics.groupby('facility_type').absolute_error.agg(['count','mean','median','max']).sort_values('mean').round(3)


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,4))
axes[0].scatter(test.occupancy_next_hour,predicted,s=4,alpha=.15,color='#0b7a70')
axes[0].plot([0,100],[0,100],'--',color='black'); axes[0].set(xlabel='Actual',ylabel='Predicted',title='Actual vs predicted')
axes[1].hist(diagnostics.absolute_error,bins=35,color='#e85d3f'); axes[1].set(title='Absolute error distribution',xlabel='Percentage points')
plt.tight_layout()


## 9. Permutation importance
Permutation importance measures the test-score reduction after shuffling one feature. It is an explanatory diagnostic, not a causal claim.

In [ ]:
sample = test.sample(min(5000,len(test)),random_state=42)
importance = permutation_importance(model,sample[FEATURE_COLUMNS],sample.occupancy_next_hour,n_repeats=5,random_state=42,scoring='neg_mean_absolute_error')
pd.Series(importance.importances_mean,index=FEATURE_COLUMNS).sort_values().plot.barh(color='#0b7a70',title='Permutation importance')
plt.xlabel('Decrease in score after permutation'); plt.tight_layout()


## 10. Artifact contract tests

In [ ]:
assert MODEL_PATH.exists() and (ROOT/'occupancy_model.joblib').exists()
assert artifact['features'] == FEATURE_COLUMNS
assert metrics['mae'] < metrics['baseline_mae']
assert metrics['r2'] > .80
assert metrics['suitable_for_clinical_use'] is False
json.loads(METRICS_PATH.read_text())


## 11. Governed-data retraining
Place an authorised de-identified CSV at `data/authorised_inventory.csv`, following `data/README.md`, then run `train_model(Path('data/authorised_inventory.csv'))`. Before a pilot, add per-hospital backtesting, drift monitoring, uncertainty intervals, fairness review, incident response, model approval and rollback procedures.

## Model card summary
**Intended use:** hackathon demonstration of operational capacity forecasting.  
**Not intended for:** diagnosis, triage, autonomous dispatch, or claims of real-time availability.  
**Primary limitation:** simulated targets cannot establish real-world performance.  
**Human oversight:** hospitals must confirm inventory and administrators must review certificate signals.  
**Refresh policy:** retrain only on governed, time-stamped inventory and compare against the persistence baseline before release.